# unet_dem_new_data: ResNet50 U-Net with the DEM on new train/val/test splits

Same model, augmentation, epochs and cutoff procedure as `unet_dem.ipynb`, trained and tested on splits that mix three sources:

- **Original scenes** 1-12 (Harvey, Houston)
- **New scenes** 13-17 and 19, labeled later (Florence NC: 13, 14, 19; Harvey west Houston: 15, 16, 17)
- **EvaNet's regions** R1-R3 (Hurricane Matthew 2016, North Carolina; loaded by `scripts/evanet_data.py`)

Pick one with `SPLIT_SET`.

**Inland generalisation** (goal: work well in inland "mid" regions, avoid coastal scenes). Coastal scenes are the low-lying ones on bays and estuaries: 4, 5, 6, 8, 9, 14, 19 and Ida (18). All other scenes count as inland. The two sets share the same inland val and test scenes and differ only in whether coastal scenes are added to training. Ida is dropped entirely.

| | `inland_plus_coastal` | `inland_only` |
|---|---|---|
| train | inland 1, 2, 3, 7, 10, 11, 16 + coastal 4, 5, 6, 8, 9, 14, 19 | inland 1, 2, 3, 7, 10, 11, 16 |
| val | 15 | 15 |
| test | 12, 13, 17 (inland) | 12, 13, 17 (inland) |

**Earlier sets** (their test scenes are mostly coastal):

| | `mixed_regions` | `mixed` (same, no regions) | `original` (yesterday's split) |
|---|---|---|---|
| train | 1, 2, 3, 8, 9 + 13, 15, 16 + R1, R3 | 1, 2, 3, 8, 9 + 13, 15, 16 | 1, 2, 3, 8, 9 |
| val | 12 + 14, 17 | 12 + 14, 17 | 12 |
| test | 4, 5, 6, 7, 10, 11 + 19 + R2 | 4, 5, 6, 7, 10, 11 + 19 | 4, 5, 6, 7, 10, 11 |
| held out | 18 (Ida) | 18 (Ida) | 18 (Ida) |

`mixed` is built from `mixed_regions` by removing the regions, so the two differ only in the regions.

`LOSS` picks the training loss: `"bce"` is `unet_dem`'s masked BCE, and `"bce+gravity"` adds EvaNet's gravity loss (water flows downhill), which uses the DEM a second time, in the loss.

Test results are reported **overall, per source** (original / new / regions) and per scene. No set ever trains, validates or tunes on Ida (18).

To fit in WSL's memory, tiles are preprocessed in place and each scene's tiles stay in their own array (never concatenated); training tiles come from a generator with a full shuffle each epoch. Weights go to `../outputs/<RUN_NAME>.weights.h5` and results to `../outputs/<RUN_NAME>_results.json`, where `RUN_NAME` is `unet_dem_<SPLIT_SET>` plus `_gravity` for the gravity loss.

**Kernel:** Python 3 (the TensorFlow environment, `tflabsfall24`).

In [1]:
import sys
import json
import copy
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras

sys.path.append("../scripts")
from dataset import get_tiles, TILE
from evanet_data import get_region_tiles

SEED = 0
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0 | Keras: 3.12.4
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
SPLIT_SET = "inland_plus_coastal"  # "inland_plus_coastal", "inland_only", "mixed_regions", "mixed" or "original" (see the tables above)
LOSS = "bce"                  # "bce" (unet_dem's loss) or "bce+gravity" (adds EvaNet's gravity loss)
GRAVITY_WEIGHT = 1.0          # weight of the gravity term when LOSS = "bce+gravity"
MAX_UNKNOWN = 1.0             # same as unet_dem.ipynb: keep every training tile
EPOCHS = 15                   # same as unet_dem.ipynb

# Coastal / low-lying scenes on bays and estuaries (median elevation 0.3-6.9 m). Everything else counts as inland.
COASTAL = {4, 5, 6, 8, 9, 14, 18, 19}
INLAND_TRAIN = [1, 2, 3, 7, 10, 11, 16]

# "scenes" are project scene numbers (data/image_N); "regions" are EvaNet's regions 1-3
SPLIT_SETS = {
    # Inland generalisation: both share the same inland val/test and differ only in the coastal training scenes
    "inland_plus_coastal": {
        "train": {"scenes": INLAND_TRAIN + [4, 5, 6, 8, 9, 14, 19], "regions": []},
        "val":   {"scenes": [15],         "regions": []},
        "test":  {"scenes": [12, 13, 17], "regions": []},
    },
    "inland_only": {
        "train": {"scenes": INLAND_TRAIN, "regions": []},
        "val":   {"scenes": [15],         "regions": []},
        "test":  {"scenes": [12, 13, 17], "regions": []},
    },
    "mixed_regions": {
        "train": {"scenes": [1, 2, 3, 8, 9, 13, 15, 16], "regions": [1, 3]},
        "val":   {"scenes": [12, 14, 17],                "regions": []},
        "test":  {"scenes": [4, 5, 6, 7, 10, 11, 19],    "regions": [2]},
    },
    "original": {
        "train": {"scenes": [1, 2, 3, 8, 9],         "regions": []},
        "val":   {"scenes": [12],                    "regions": []},
        "test":  {"scenes": [4, 5, 6, 7, 10, 11],    "regions": []},
    },
}
# "mixed" = "mixed_regions" with every region removed, so the two differ only in the regions
SPLIT_SETS["mixed"] = copy.deepcopy(SPLIT_SETS["mixed_regions"])
for part in SPLIT_SETS["mixed"].values():
    part["regions"] = []

# Ida (18) is dropped entirely for the inland sets; the older sets still score it as a held-out scene
HELD_OUT = [] if SPLIT_SET.startswith("inland") else [18]
NEW_SCENES = {13, 14, 15, 16, 17, 19}

# No scene or region may appear in two splits, and Ida never in any split
S = SPLIT_SETS[SPLIT_SET]
used = [("scene", i) for p in S.values() for i in p["scenes"]] + [("region", r) for p in S.values() for r in p["regions"]]
assert len(used) == len(set(used)), "a scene or region is in more than one split"
assert 18 not in {i for p in S.values() for i in p["scenes"]}, "Ida (18) must never be trained, validated or tested on"
if SPLIT_SET.startswith("inland"):
    assert not COASTAL & set(S["val"]["scenes"] + S["test"]["scenes"]), "inland val/test must not contain coastal scenes"
assert LOSS in ("bce", "bce+gravity")

RUN_NAME = f"unet_dem_{SPLIT_SET}" + ("_gravity" if LOSS == "bce+gravity" else "")
WEIGHTS = Path(f"../outputs/{RUN_NAME}.weights.h5")
RESULTS = Path(f"../outputs/{RUN_NAME}_results.json")
print("run:", RUN_NAME, "| loss:", LOSS)
for split, p in S.items():
    tags = ", ".join(f"{i}{'(c)' if i in COASTAL else ''}" for i in p["scenes"])
    print(f"  {split:5}  scenes {tags}  regions {p['regions']}")
print(f"  held out: {HELD_OUT or 'none'}    (c) = coastal")

run: unet_dem_inland_plus_coastal | loss: bce
  train  scenes 1, 2, 3, 7, 10, 11, 16, 4(c), 5(c), 6(c), 8(c), 9(c), 14(c), 19(c)  regions []
  val    scenes 15  regions []
  test   scenes 12, 13, 17  regions []
  held out: none    (c) = coastal


In [3]:
def source(unit):
    """'regions', 'new' or 'original' for a unit name like 'R2' or 'image_14'."""
    if unit.startswith("R"):
        return "regions"
    return "new" if int(unit.split("_")[1]) in NEW_SCENES else "original"


def load_units(split, stride, max_unknown=1.0):
    """{unit: (X, y)} for every scene and region in a split, one array pair per unit."""
    units = {f"image_{i}": get_tiles([i], stride=stride, max_unknown=max_unknown) for i in S[split]["scenes"]}
    for r in S[split]["regions"]:
        units[f"R{r}"] = get_region_tiles([r], stride=stride, max_unknown=max_unknown)
    return units


# Same tiling as unet_dem.ipynb: train overlaps by half, val and test keep every non-overlapping tile
train_units = load_units("train", stride=128, max_unknown=MAX_UNKNOWN)
val_units = load_units("val", stride=TILE)
test_units = load_units("test", stride=TILE)

for split, units in [("train", train_units), ("val", val_units), ("test", test_units)]:
    print(f"{split}: {sum(len(X) for X, _ in units.values())} tiles")
    for u, (X, y) in units.items():
        known = y[y != -1]
        print(f"   {u:9} {source(u):8}  {len(X):4} tiles  unknown {(y == -1).mean():5.1%}  flood (of known) {known.mean():5.1%}")

# Training stays as separate arrays (no concatenation, which would need twice the RAM); val is small
train_parts = list(train_units.values())
X_val = np.concatenate([X for X, _ in val_units.values()])
y_val = np.concatenate([y for _, y in val_units.values()])
del val_units

train: 3143 tiles
   image_1   original   221 tiles  unknown 14.5%  flood (of known) 76.1%
   image_2   original   352 tiles  unknown 18.9%  flood (of known) 97.3%
   image_3   original   600 tiles  unknown 32.9%  flood (of known) 91.6%
   image_7   original   112 tiles  unknown 15.3%  flood (of known) 35.5%
   image_10  original   195 tiles  unknown 55.2%  flood (of known) 64.0%
   image_11  original   180 tiles  unknown 39.6%  flood (of known) 85.4%
   image_16  new         21 tiles  unknown 23.0%  flood (of known) 96.6%
   image_4   original   192 tiles  unknown 71.9%  flood (of known) 96.6%
   image_5   original   110 tiles  unknown 75.7%  flood (of known) 99.7%
   image_6   original   121 tiles  unknown 16.5%  flood (of known) 100.0%


   image_8   original   299 tiles  unknown 33.0%  flood (of known) 11.5%
   image_9   original   500 tiles  unknown 14.4%  flood (of known)  5.9%
   image_14  new        120 tiles  unknown 17.5%  flood (of known) 70.1%
   image_19  new        120 tiles  unknown 21.8%  flood (of known) 25.6%
val: 120 tiles
   image_15  new        120 tiles  unknown 14.0%  flood (of known) 77.3%
test: 182 tiles
   image_12  original   110 tiles  unknown 20.0%  flood (of known) 51.3%
   image_13  new         40 tiles  unknown 21.3%  flood (of known) 24.1%
   image_17  new         32 tiles  unknown 31.2%  flood (of known) 35.1%


## Prepare the images for ResNet50

Same as `unet_dem.ipynb`: RGB goes to 0-255 and then `preprocess_input`, and the per-image-centred DEM is multiplied by `DEM_SCALE = 20`. It's done in place in chunks, so no second copy of the data is made.

In [4]:
from tensorflow.keras.applications.resnet50 import preprocess_input

DEM_SCALE = 20  # same as unet_dem.ipynb


def preprocess_inplace(X, chunk=256):
    """Same result as unet_dem.ipynb's preprocess(), written back into X."""
    for s in range(0, len(X), chunk):
        c = X[s:s + chunk]
        c[..., :3] = preprocess_input(c[..., :3] * 255.0)
        c[..., 3] *= DEM_SCALE
    return X


for X, _ in train_parts:
    preprocess_inplace(X)
preprocess_inplace(X_val)
for X, _ in test_units.values():
    preprocess_inplace(X)

X0 = train_parts[0][0]
print("RGB min/max:", X0[..., :3].min(), X0[..., :3].max())
print("DEM min/max:", X0[..., 3].min(), X0[..., 3].max())

RGB min/max: -123.68 151.061
DEM min/max: -257.7552 124.98047


## Loss functions (skip unknown pixels)

- `"bce"`: `unet_dem.ipynb`'s masked BCE, unchanged.
- `"bce+gravity"`: masked BCE + `GRAVITY_WEIGHT` × EvaNet's gravity loss, rewritten in TensorFlow from `scripts/evanet/loss.py`. Each pixel is compared with its 3×3 neighbours:
  - a **flooded** neighbour at the same height or **higher** means this pixel should be flood
  - a **dry** neighbour at the same height or **lower** means this pixel should be dry
  - other pairs and unknown neighbours are ignored

  It's averaged per pixel pair (pixels × 9), so it is on the same scale as BCE, as in `evanet_resnet.ipynb`'s `"gravity+bce"`. The U-Net has one sigmoid output `p`, so EvaNet's signed prediction is `p` when the pixel is predicted flood (`p > 0.5`) and `-(1 - p)` when predicted dry. The DEM it needs is passed in as a second channel of `y_true`, so it gets the same rotations and flips as the image.

In [5]:
def masked_bce(y_true, y_pred):
    """unet_dem.ipynb's loss. Uses channel 0 of y_true (the label: 1 flood, 0 dry, -1 unknown)."""
    y_true = y_true[..., :1]
    known = tf.cast(y_true != -1, "float32")
    y_true = tf.where(y_true == -1, 0.0, y_true)
    loss = keras.losses.binary_crossentropy(y_true, y_pred)[..., None]
    return tf.reduce_sum(loss * known) / tf.reduce_sum(known)


def _neighbours(t):
    """(B, H, W, 1) -> (B, H, W, 9): each pixel's 3x3 window, zero-padded at the edges (like torch Unfold)."""
    return tf.image.extract_patches(t, sizes=[1, 3, 3, 1], strides=[1, 1, 1, 1], rates=[1, 1, 1, 1], padding="SAME")


def gravity_loss(y_true, y_pred):
    """EvaNet's ElevationLoss (scripts/evanet/loss.py) in TensorFlow, averaged per pixel pair.
    y_true channel 0 = label (1 / 0 / -1), channel 1 = DEM (only height differences matter)."""
    label, dem = y_true[..., :1], y_true[..., 1:2]
    gt = tf.where(label == 1, 1.0, tf.where(label == 0, -1.0, 0.0))  # flood +1, dry -1, unknown 0
    signed = tf.where(y_pred > 0.5, y_pred, y_pred - 1.0)            # p if predicted flood, -(1 - p) if dry
    gt_n, dem_n = _neighbours(gt), _neighbours(dem)
    score = 1.0 - gt_n * signed                                      # (B, H, W, 9)
    higher = tf.cast(dem > dem_n, "float32")                         # this pixel above the neighbour
    lower = tf.cast(dem < dem_n, "float32")                          # this pixel below the neighbour
    flood_n = tf.cast(gt_n == 1, "float32")
    dry_n = tf.cast(gt_n == -1, "float32")
    applies = tf.cast(gt_n != 0, "float32") * (1.0 - flood_n * higher) * (1.0 - dry_n * lower)
    return tf.reduce_sum(applies * score) / tf.cast(tf.size(gt_n), "float32")


def bce_gravity(y_true, y_pred):
    return masked_bce(y_true, y_pred) + GRAVITY_WEIGHT * gravity_loss(y_true, y_pred)


loss_fn = masked_bce if LOSS == "bce" else bce_gravity

# Quick check on a toy 1 x 3 x 3 tile: the centre pixel sits lowest next to flooded neighbours,
# so predicting it dry (0.1) must cost more gravity loss than predicting it flood (0.9).
_lab = tf.constant([[1, 1, 1], [1, -1, 1], [1, 1, 1]], "float32")[None, ..., None]
_dem = tf.constant([[2, 2, 2], [2, 0, 2], [2, 2, 2]], "float32")[None, ..., None]
_yt = tf.concat([_lab, _dem], -1)
_dry = tf.fill([1, 3, 3, 1], 0.9) * tf.constant([[1, 1, 1], [1, 0.111, 1], [1, 1, 1]])[None, ..., None]
_wet = tf.fill([1, 3, 3, 1], 0.9)
print(f"gravity loss, low centre predicted dry {gravity_loss(_yt, _dry):.4f} > predicted flood {gravity_loss(_yt, _wet):.4f}")
assert gravity_loss(_yt, _dry) > gravity_loss(_yt, _wet)
print("loss:", LOSS)

gravity loss, low centre predicted dry 0.2272 > predicted flood 0.0494
loss: bce


## Build the model: ResNet50 encoder

Same as `unet_dem.ipynb` (from https://github.com/nikhilroxtomar/Semantic-Segmentation-Architecture/blob/main/TensorFlow/resnet50_unet.py): ImageNet weights, plus a 4th DEM input to `conv1` that starts at zero.

In [6]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50


def conv_block(x, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def decoder_block(x, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, 2, strides=2, padding="same")(x)
    x = Concatenate()([x, skip_features])
    return conv_block(x, num_filters)


def build_resnet50_unet(input_shape):
    inputs = Input(input_shape)
    resnet50 = ResNet50(include_top=False, weights=None, input_tensor=inputs)

    pretrained = ResNet50(include_top=False, weights="imagenet", input_shape=(256, 256, 3))
    for layer, src in zip(resnet50.layers, pretrained.layers):
        weights = src.get_weights()
        if layer.name == "conv1_conv":
            kernel, bias = weights
            weights = [np.concatenate([kernel, np.zeros_like(kernel[:, :, :1])], axis=2), bias]
        layer.set_weights(weights)

    s1 = inputs
    s2 = resnet50.get_layer("conv1_relu").output
    s3 = resnet50.get_layer("conv2_block3_out").output
    s4 = resnet50.get_layer("conv3_block4_out").output
    b1 = resnet50.get_layer("conv4_block6_out").output

    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return Model(inputs, outputs, name="ResNet50_U-Net"), resnet50


model, encoder = build_resnet50_unet((256, 256, 4))
print(model.input_shape, "->", model.output_shape)

(None, 256, 256, 4) -> (None, 256, 256, 1)


## Train

Same settings as `unet_dem.ipynb`: frozen encoder except `conv1`, Adam 1e-3, batch 4, 15 epochs, random rot90 and flip, and the best `val_loss` epoch is saved (val loss is pooled over every val scene and uses the selected `LOSS`). Training tiles come from a generator that reshuffles every tile each epoch.

In [7]:
encoder.trainable = False
encoder.get_layer("conv1_conv").trainable = True

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss_fn, jit_compile=False)

WEIGHTS.parent.mkdir(exist_ok=True)
checkpoint = keras.callbacks.ModelCheckpoint(str(WEIGHTS), monitor="val_loss",
                                             save_best_only=True, save_weights_only=True)


def augment(x, y):
    """Same as unet_dem.ipynb: random rot90, then a 50% left-right flip (tf.cond instead of a Python if).
    y may carry the DEM as a 2nd channel; it is rotated and flipped together with the label."""
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x, y = tf.image.rot90(x, k), tf.image.rot90(y, k)
    flip = tf.random.uniform([]) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    y = tf.cond(flip, lambda: tf.image.flip_left_right(y), lambda: y)
    return x, y


# With the gravity loss, y_true = [label, DEM]; with BCE alone it is just the label (as in unet_dem)
Y_CH = 2 if LOSS == "bce+gravity" else 1


def target(X, y, i):
    lab = y[i].astype(np.float32)[..., None]
    return np.concatenate([lab, X[i][..., 3:4]], axis=-1) if Y_CH == 2 else lab


index = [(p, i) for p, (X, _) in enumerate(train_parts) for i in range(len(X))]
rng = np.random.default_rng(SEED)


def train_tiles():
    for j in rng.permutation(len(index)):
        p, i = index[j]
        X, y = train_parts[p]
        yield X[i], target(X, y, i)


signature = (tf.TensorSpec((256, 256, 4), tf.float32), tf.TensorSpec((256, 256, Y_CH), tf.float32))
train_ds = (tf.data.Dataset.from_generator(train_tiles, output_signature=signature)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(4).prefetch(2))


def val_tiles():
    for i in range(len(X_val)):
        yield X_val[i], target(X_val, y_val, i)


val_ds = tf.data.Dataset.from_generator(val_tiles, output_signature=signature).batch(4)

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint], verbose=2)

Epoch 1/15


E0000 00:00:1795035959.502029   18083 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/gradient_tape/compile_loss/bce_gravity/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


/home/jdcont/miniconda3/envs/tflabsfall24/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


630/630 - 84s - 133ms/step - loss: 0.6204 - val_loss: 0.7418


Epoch 2/15


630/630 - 67s - 107ms/step - loss: 0.5039 - val_loss: 0.8967


... (22 outputs left out)

Epoch 14/15


630/630 - 67s - 107ms/step - loss: 0.1716 - val_loss: 0.4167


Epoch 15/15


630/630 - 67s - 107ms/step - loss: 0.1682 - val_loss: 0.5278


## Results

Same metrics as `unet_dem.ipynb`: flood and dry IoU over known pixels, their mean, and the share predicted flood. The cutoff is picked on **val** (all val scenes pooled) from 0.3-0.7 by the mean of flood and dry IoU, and test is scored at that cutoff and at 0.5.

Test is reported pooled over everything, then per source (the "original scenes" group is yesterday's test set), then per scene or region, then Ida.

In [8]:
model.load_weights(str(WEIGHTS))


def counts(probs, y, cut):
    """Pixel counts over known pixels: flood inter/union, dry inter/union, predicted flood, known, labelled flood."""
    pf, k = probs > cut, y != -1
    tf_, td = y == 1, y == 0
    return np.array([(pf & tf_ & k).sum(), ((pf | tf_) & k).sum(), (~pf & td & k).sum(),
                     ((~pf | td) & k).sum(), (pf & k).sum(), k.sum(), (tf_ & k).sum()], dtype=np.int64)


def summary(c):
    """Flood IoU, dry IoU, their mean, share predicted flood, share labelled flood."""
    f, d = c[0] / max(c[1], 1), c[2] / max(c[3], 1)
    return {"flood_iou": float(f), "dry_iou": float(d), "mean_iou": float((f + d) / 2),
            "says_flood": float(c[4] / c[5]), "label_flood": float(c[6] / c[5])}


def line(name, s):
    return (f"{name:22} flood IoU {s['flood_iou']:.3f}  dry IoU {s['dry_iou']:.3f}  mean {s['mean_iou']:.3f}  "
            f"says flood {s['says_flood']:5.1%}  label {s['label_flood']:5.1%}")


def predict(X):
    return model.predict(X, batch_size=8, verbose=0)[..., 0]


# Cutoff from val (all val scenes pooled)
val_probs = predict(X_val)
sweep = {cut: summary(counts(val_probs, y_val, cut)) for cut in [0.3, 0.4, 0.5, 0.6, 0.7]}
for cut, s in sweep.items():
    print(line(f"val cutoff {cut}", s))
CUTOFF = max(sweep, key=lambda c: sweep[c]["mean_iou"])
print("chosen cutoff:", CUTOFF)

# Predict each test unit once, plus the held-out scene(s)
probs = {u: (predict(X), y) for u, (X, y) in test_units.items()}
held = {}
for i in HELD_OUT:
    Xi, yi = get_tiles([i])
    held[f"image_{i}"] = (predict(preprocess_inplace(Xi)), yi)

groups = {"all test": list(probs)}
for src, label in [("original", "original scenes"), ("new", "new scenes"), ("regions", "EvaNet regions")]:
    members = [u for u in probs if source(u) == src]
    if members:
        groups[label] = members

results = {"run": RUN_NAME, "split_set": SPLIT_SET, "splits": S, "held_out": HELD_OUT, "cutoff": CUTOFF,
           "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
           "val": {str(c): s for c, s in sweep.items()}, "test": {}}
for cut in sorted({CUTOFF, 0.5}):
    per_unit = {u: counts(p, y, cut) for u, (p, y) in probs.items()}
    r = {"groups": {g: summary(sum(per_unit[u] for u in m)) for g, m in groups.items()},
         "per_unit": {u: summary(c) for u, c in per_unit.items()},
         "held_out": {u: summary(counts(p, y, cut)) for u, (p, y) in held.items()}}
    results["test"][str(cut)] = r
    print(f"\n===== TEST at cutoff {cut}" + ("  (chosen on val)" if cut == CUTOFF else ""))
    for g, s in r["groups"].items():
        print(line(g, s))
    print()
    for u, s in r["per_unit"].items():
        print(line(f"  {u} ({source(u)})", s))
    for u, s in r["held_out"].items():
        print(line(f"  {u} (Ida, held out)", s))

RESULTS.write_text(json.dumps(results, indent=1))
print("\nsaved", RESULTS)

val cutoff 0.3         flood IoU 0.764  dry IoU 0.683  mean 0.724  says flood 65.6%  label 51.5%
val cutoff 0.4         flood IoU 0.802  dry IoU 0.748  mean 0.775  says flood 61.9%  label 51.5%
val cutoff 0.5         flood IoU 0.831  dry IoU 0.796  mean 0.813  says flood 58.9%  label 51.5%
val cutoff 0.6         flood IoU 0.854  dry IoU 0.833  mean 0.843  says flood 55.8%  label 51.5%
val cutoff 0.7         flood IoU 0.875  dry IoU 0.864  mean 0.870  says flood 52.8%  label 51.5%
chosen cutoff: 0.7



===== TEST at cutoff 0.5
all test               flood IoU 0.824  dry IoU 0.644  mean 0.734  says flood 71.8%  label 66.6%
original scenes        flood IoU 0.841  dry IoU 0.535  mean 0.688  says flood 79.6%  label 76.0%
new scenes             flood IoU 0.638  dry IoU 0.833  mean 0.735  says flood 35.5%  label 23.0%

  image_4 (original)   flood IoU 0.982  dry IoU 0.554  mean 0.768  says flood 96.2%  label 97.6%
  image_5 (original)   flood IoU 0.996  dry IoU 0.613  mean 0.805  says flood 99.1%  label 99.4%
  image_6 (original)   flood IoU 0.834  dry IoU 0.000  mean 0.417  says flood 83.4%  label 100.0%
  image_7 (original)   flood IoU 0.541  dry IoU 0.520  mean 0.530  says flood 65.8%  label 37.0%
  image_10 (original)  flood IoU 0.865  dry IoU 0.761  mean 0.813  says flood 66.4%  label 64.1%
  image_11 (original)  flood IoU 0.935  dry IoU 0.660  mean 0.798  says flood 86.0%  label 85.8%
  image_19 (new)       flood IoU 0.638  dry IoU 0.833  mean 0.735  says flood 35.5%  label 23.0%
  